## 项目目录结构

```
libtorch_demo/
├── libtorch/                      # CPU 版 libtorch 库（从 PyTorch 官网下载，无需修改）
│   ├── include/                   #   C++ 头文件
│   ├── lib/                       #   .so 动态链接库
│   └── share/cmake/Torch/         #   CMake 配置文件（TorchConfig.cmake）
│
├── demo/
│   ├── pytorch/
│   │   ├── export_to_pt.py        # [Python] 定义模型 → 导出 TorchScript (.pt)
│   │   └── model.pt               #   导出的模型文件（运行脚本后生成）
│   │
│   └── cpp/
│       ├── CMakeLists.txt         # [构建] CMake 编译配置，链接 libtorch
│       ├── main.cpp               # [C++]  加载 .pt 模型 → 构造输入 → 推理 → 输出结果
│       └── build/                 #   编译输出目录（cmake/make 自动创建）
│           └── libtorch_demo      #     编译生成的可执行文件
│
└── 运行步骤.ipynb                  # 本 notebook
```

## 各文件简要说明

| 文件 | 作用 |
|------|------|
| `export_to_pt.py` | 定义一个简单的 3 层 MLP（输入4维 → 隐藏层16维 → 输出3维），用 `torch.jit.trace` 导出为 TorchScript 格式 |
| `model.pt` | 导出后的模型文件，包含网络结构和权重，C++ 端直接加载使用 |
| `CMakeLists.txt` | 通过 `CMAKE_PREFIX_PATH` 指向 `libtorch/`，调用 `find_package(Torch)` 自动配置头文件和库路径 |
| `main.cpp` | C++ 推理程序：加载模型 → 构造输入 tensor → 执行 forward → 打印输出，并演示逐元素访问和多次推理 |

# LibTorch Demo 运行步骤

本 notebook 记录从 Python 导出模型到 C++ 加载推理的完整流程。

## 整体流程

1. **Python 端**：用 `torch.jit.trace` 将 PyTorch 模型导出为 TorchScript (`.pt` 文件)
2. **C++ 端**：用 `torch::jit::load` 加载 `.pt` 文件并执行推理

---
## Step 1：Python 端导出模型

先看一下我们要导出的模型定义：

In [ ]:
# 查看 export_to_pt.py 的内容
!cat demo/pytorch/export_to_pt.py

运行导出脚本，会在 `demo/pytorch/` 下生成 `model.pt`：

In [ ]:
import os
os.chdir('demo/pytorch')
!python export_to_pt.py
os.chdir('../..')

确认 `model.pt` 文件已生成：

In [ ]:
!ls -lh demo/pytorch/model.pt

---
## Step 2：C++ 端编译

### 2.1 查看 CMakeLists.txt

CMake 配置会自动通过相对路径找到 `libtorch/` 目录：

In [ ]:
!cat demo/cpp/CMakeLists.txt

### 2.2 查看 main.cpp

In [ ]:
!cat demo/cpp/main.cpp

### 2.3 编译

In [ ]:
!mkdir -p demo/cpp/build
!cd demo/cpp/build && cmake .. && make

---
## Step 3：运行 C++ 推理程序

In [ ]:
!cd demo/cpp/build && ./libtorch_demo ../../pytorch/model.pt

---
## 关键概念对照表

| 概念 | Python 端 | C++ 端 |
|------|-----------|--------|
| 导出方式 | `torch.jit.trace(model, dummy_input)` | — |
| 加载模型 | `torch.jit.load(path)` | `torch::jit::load(path)` |
| 推理调用 | `model(input)` | `module.forward({input}).toTensor()` |
| 关闭梯度 | `with torch.no_grad():` | `torch::NoGradGuard no_grad;` |
| 头文件 | — | 只需 `#include <torch/script.h>` |

## 常见问题

**Q: 编译时找不到 libtorch？**  
A: 修改 `CMakeLists.txt` 中的 `CMAKE_PREFIX_PATH` 为绝对路径：
```cmake
set(CMAKE_PREFIX_PATH "/你的绝对路径/libtorch_demo/libtorch")
```

**Q: trace 和 script 的区别？**  
A: `trace` 通过记录一次前向传播的计算图来导出，适合没有复杂控制流的模型；`script` 直接编译 Python 代码，支持 if/for 等动态控制流。简单模型推荐 `trace`。

**Q: 如何加载带训练权重的模型？**  
A: 先在 Python 端 `model.load_state_dict(torch.load('weights.pth'))` 加载权重，再 `trace` 导出即可，权重会被一起打包进 `.pt` 文件。